In [1]:
import pandas as pd
data = pd.read_csv(r"tested.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          332 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         417 non-null    float64
 10  Cabin        91 non-null     object 
 11  Embarked     418 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 39.3+ KB


In [ ]:
data = data.drop("Cabin", axis= 1)
data.info()

In [ ]:
data.info()

In [ ]:
data["Age"] = data["Age"].fillna(data["Age"].median())
data.info()

In [ ]:
data["Fare"] = data["Fare"].fillna(data["Fare"].mean())
data.info()

In [ ]:
#droping columns passengerid, name, ticket, embarked
data.drop(columns= ["PassengerId", "Name", "Ticket", "Embarked"], inplace= True)
#data.info()
data.head()
data.drop(columns= [ "Fare"], inplace= True)
data.info()

In [ ]:
data.head()

In [ ]:
invalid_sex = data.loc[~data["Sex"].isin(["male", "female"]), "Sex"]
print("Invalid entries found:", invalid_sex.unique())

In [ ]:
data.loc[data["Sex"] == "male", "Sex"] = 1
data.loc[data["Sex"] == "female", "Sex"] = 0

In [ ]:
data.head()
x = []
for i in data["Sex"] :
    x.append(i)

data["Sex"] = x
data.info()
data.head()

In [ ]:
# Step 1: Correlation analysis
corr_matrix = data.corr(numeric_only=True)
correlations_with_target = corr_matrix["Survived"].sort_values(ascending=False)

# Display numeric correlations
correlations_with_target

In [ ]:
# Step 2: Plot correlation heatmap
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
#features and target
# x - features, y - target var
x = data.drop(columns=["Survived"])
y = data["Survived"]
x.head()
y.head()

In [ ]:
from sklearn.model_selection import train_test_split

# First split into training+validation and test
x_temp, x_test, y_temp, y_test = train_test_split(x, y, test_size=0.15, random_state=42)

# Then split training+validation into separate train and validation sets
x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=0.1765, random_state=42)
# 0.1765 of 85% ≈ 15% of total (so: 70/15/15)

In [ ]:
#training the model - using logistic regression
from sklearn.linear_model import LogisticRegression
model_titanic = LogisticRegression(max_iter = 1000)
model_titanic.fit(x_train, y_train)

In [ ]:
# Evaluate model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
y_pred = model_titanic.predict(x_test)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred)
}
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(  f"Precision: {precision_score(y_test, y_pred)}")
print(    f"Recall: {recall_score(y_test, y_pred)}")
print(    f"F1 Score: { f1_score(y_test, y_pred)}")

In [ ]:
# Step 4: Feature importance (coefficients)
feature_importance = pd.Series(model_titanic.coef_[0], index=x.columns).sort_values(ascending=False)

# Plot feature importance
plt.figure(figsize=(8, 6))
feature_importance.plot(kind="barh")
plt.title("Feature Importance (Logistic Regression Coefficients)")
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.show()

In [ ]:
new_predictions = model_titanic.predict(x_val)
new_predictions == y_val # returns all true - inline with the evaluation metrics above (model is 100% accurate)
#notice the heatmap where sex affected survival rate inversely 100% (-1.00)